# **🎥 Clean & Jerk Dataset — Post-Export Video Blur Fixer (Publication QA)**

Fix missed blur regions (extra people, mirrors, background clutter) directly on **already-exported MP4 videos** on Google Drive (Colab) or local disk without re-exporting from raw sources.

**Pipeline & Guarantees:**
`Exported /videos/*.mp4` → **Inline HTML5 Canvas Editor (Colab) / PySide6 Pop-out (Local)** → **FFmpeg Pipe (`libx264`, `yuv420p`, CRF 17, exact CFR FPS)** → **`ffprobe` Exact Frame-Count Assertion** → **Original Backed Up (`_originals_backup/`) + Atomic Replace** → **`checksums.sha256` Update**

### 🧱 4 Mask Layers (Zero Frame-by-Frame Drudgery)
1. **Global (All Frames):** Paint once → blurs every frame (static background, mirrors, seated bystanders).
2. **Range `[In..Out]`:** Click **Set In (I)** & **Set Out (O)**, paint once → blurs only frames `In..Out`.
3. **Moving Keyframe:** Click/drag on frame A and frame B → smoothly interpolates a moving blur circle across frames (supports multiple tracks: `+ New Track`).
4. **Single Frame:** Exact brush/eraser override on the current frame only.

### 1. 🔧 Colab Setup, Drive Mount & Library Init

In [8]:
# @title 🔧 Colab Setup & Library Init { display-mode: "form" }
import importlib, os, subprocess, sys, types
from pathlib import Path

if "imp" not in sys.modules:
    try:
        import imp  # noqa: F401
    except ModuleNotFoundError:
        _imp = types.ModuleType("imp")
        _imp.reload = importlib.reload
        sys.modules["imp"] = _imp

%load_ext autoreload
%autoreload 2

IN_COLAB = False
try:
    from google.colab import drive
    IN_COLAB = True
    if not os.path.exists("/content/drive/MyDrive"):
        try:
            drive.mount("/content/drive", force_remount=False)
        except Exception as e:
            print(f"⚠️ Drive not mounted ({e}). Mount via Colab sidebar if needed.")
except ImportError:
    pass

repo_dir = "/content/Wearable-Biomechanics" # @param {type:"string"}
git_branch = "dataset_explorer" # @param {type:"string"}

if os.path.exists(os.path.join(".", "src", "export", "video_blur_colab.py")):
    repo_root = os.path.abspath(".")
else:
    if not os.path.exists(repo_dir):
        subprocess.check_call([
            "git", "clone", "-b", git_branch,
            "https://github.com/janithcyapa/Wearable-Biomechanics.git", repo_dir
        ])
    else:
        subprocess.run(["git", "-C", repo_dir, "checkout", git_branch], check=False)
        subprocess.run(["git", "-C", repo_dir, "pull", "--ff-only"], check=False)
        for _m in [k for k in list(sys.modules) if k == "src" or k.startswith("src.")]:
            importlib.reload(sys.modules[_m])
    repo_root = os.path.abspath(repo_dir)

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import pandas as pd
from IPython.display import display

DEFAULT_CLEAN_DATASET_DIR = (
    "/content/drive/MyDrive/Academic/Research/Wearable Sensing/Wearable Biomechanics/Clean&JerkDataset"
)
from src.export import (
    BACKUP_DIR_NAME,
    launch_blur_editor,
    launch_colab_blur_editor,
    scan_videos_folder,
    set_video_qa_status,
    update_checksums_sha256,
    verify_all_frame_counts,
)

print(f"Runtime: {'Google Colab (Inline HTML5 Canvas)' if IN_COLAB else 'Local Python (PySide6 Pop-out)'}")
print(f"Repo root: {repo_root}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


ModuleNotFoundError: No module named 'src.export'

### 2. Configure Exported Dataset Path & Scan QA Status

In [ ]:
DATASET_DIR = Path(DEFAULT_CLEAN_DATASET_DIR)  # Override if stored elsewhere
VIDEOS_DIR = DATASET_DIR / "videos"

videos_df = scan_videos_folder(VIDEOS_DIR, probe_all=False)
print("Status counts:")
display(videos_df["status"].value_counts().to_frame("count"))
display(videos_df)

### 3. Launch Interactive Video Blur Editor
- **In Colab:** Opens the **inline HTML5 Canvas Brush Editor** directly below this cell.
- **On Local Linux:** Opens the **PySide6 pop-out window**.
- Set `TARGET_VIDEO = None` to automatically open the first **`Pending`** video, or pass a filename like `"set_0012_front.mp4"`.

In [ ]:
TARGET_VIDEO = None  # e.g. "set_0012_front.mp4" or None to open first Pending video

if TARGET_VIDEO is not None:
    target_path = VIDEOS_DIR / TARGET_VIDEO
else:
    pending = videos_df[videos_df["status"] == "Pending"]
    if len(pending) > 0:
        target_path = VIDEOS_DIR / pending.iloc[0]["filename"]
    else:
        target_path = VIDEOS_DIR

print(f"Opening editor on: {target_path}")
if IN_COLAB:
    launch_colab_blur_editor(target_path)
else:
    launch_blur_editor(target_path, wait=True)

### 4. Publication Integrity Audit (`ffprobe` Frame-Count & Codec Verification)
Run after finishing edits in Section 3. Verifies every fixed video in `VIDEOS_DIR` has the **exact same frame count** as its pristine original in `_originals_backup/`, plus `h264` / `yuv420p` compliance.

In [ ]:
audit_df = verify_all_frame_counts(VIDEOS_DIR)
if len(audit_df) == 0:
    print("No fixed/backed-up videos found yet.")
else:
    all_match = bool(audit_df["frame_match"].all())
    print(f"Fixed videos audited: {len(audit_df)} | All frame counts exact match: {all_match}")
    display(audit_df)

pending_df = scan_videos_folder(VIDEOS_DIR)
display(pending_df["status"].value_counts().to_frame("count"))
remaining = pending_df[pending_df["status"] == "Pending"]
print(f"Remaining Pending videos: {len(remaining)}")
if len(remaining) > 0:
    display(remaining[["idx", "filename", "status"]])

### 5. Update `checksums.sha256` (Run After All Fixes Are Verified)
Updates only the modified video entries inside `DATASET_DIR/checksums.sha256`. Remember to move/delete `_originals_backup/` and `_blur_recipes/` from `VIDEOS_DIR` before final dataset packaging/upload.

In [ ]:
RUN_CHECKSUM_UPDATE = False  # Set to True once all fixed videos pass Section 4

if RUN_CHECKSUM_UPDATE:
    summary = update_checksums_sha256(DATASET_DIR, videos_subdir=VIDEOS_DIR.name)
    print("checksums.sha256 updated:", summary)
else:
    print("Set RUN_CHECKSUM_UPDATE = True to refresh checksums.sha256.")